In [2]:
!pip install pyspark
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Prova_esame")
sc = SparkContext(conf=conf)

In [25]:
CatalogueRDD=sc.textFile("./data/Catalogue.txt")
DailySalesRDD=sc.textFile("./data/DailySales.txt")
outputPath1="./output1_v2/"

#Task 1

In [26]:
CategoriesRDD=CatalogueRDD.map(lambda x: (x.split(",")[0],x.split(",")[2]))


Sales2223RDD = DailySalesRDD \
    .map(lambda x: x.split(",")) \
    .map(lambda x: ((x[0], x[1].split("/")[0]), int(x[2]))) \
    .filter(lambda x: x[0][1] in ('2022', '2023')) \
    .reduceByKey(lambda a, b: a + b) \
    .map(lambda x: (x[0][0], (0, x[1])) if x[0][1]=='2023' else (x[0][0], (x[1], 0))) \
    .reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1]))

joinedRDD=Sales2223RDD.join(CategoriesRDD).map(lambda x: (x[1][1],x[1][0])) \
    .reduceByKey(lambda v1,v2: (v1[0]+v2[0],v1[1]+v2[1])) \
    .filter(lambda x: x[1][1]>x[1][0]) \
    .keys()

joinedRDD.saveAsTextFile(outputPath1)

#Task 2

In [27]:
PricesRDD=sc.textFile("./data/Prices.txt")
outputPath2="./outputPath2/"

In [53]:
from datetime import timedelta, datetime
def nextDate(date):
    date_obj = datetime.strptime(date, '%Y/%m/%d')
    prev_date = date_obj + timedelta(days=1)
    return prev_date.strftime('%Y/%m/%d')

In [69]:
allSalesRDD=DailySalesRDD.map(lambda x: (x.split(",")[0],(x.split(",")[1],int(x.split(",")[2]))))
cleanedPricesRDD=PricesRDD.map(lambda x: (x.split(",")[0],((x.split(",")[1],x.split(",")[2]),float(x.split(",")[3]))))
newjoinedRDD=allSalesRDD.join(cleanedPricesRDD)

def countMoney(row):
  purchaseDate=row[1][0][0]
  startPrice=row[1][1][0][0]
  endPrice=row[1][1][0][1]
  price=row[1][1][1]
  numItems=row[1][0][1]
  key=row[0]
  if (purchaseDate<=endPrice and purchaseDate>=startPrice):
    dayCash=price*numItems
    return ((key,purchaseDate),dayCash)



moneyDayRDD=newjoinedRDD.map(countMoney).filter(lambda x: x is not None) \
    .sortByKey().map(lambda x: (x[0][0], (x[0][1], x[1]))).groupByKey()

def increasing_dates(record):
    item, values = record
    result = []
    prev_income = None
    for date, income in values:
        if prev_income is not None and income > prev_income:
            result.append((item, date))
        prev_income = income
    return result

increasingRDD = moneyDayRDD.flatMap(increasing_dates)
increasingRDD.saveAsTextFile(outputPath2)